# End-to-End Retrieval-Augmented Generation


In [1]:
import torch

# GPU enable for the session
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

CUDA available: False
GPU count: 0


## Processing the Data


In [6]:
import urllib.request
import os

# Define the file name and URL
file_name = "The-AI-Act.pdf"
url = "https://artificialintelligenceact.eu/wp-content/uploads/2021/08/The-AI-Act.pdf"

# Download the file
if not os.path.exists(file_name):
    urllib.request.urlretrieve(url, file_name)
    print(f"{file_name} downloaded successfully.")    
else:
    print(f"{file_name} already exists.")

The-AI-Act.pdf already exists.


In [ ]:
#!pip install langchain_community pypdf langchain-text-splitters

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_name)
docs = loader.load()
print(len(docs))                    # total number of pages

W0505 08:40:01.709000 85076 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.



108


In [15]:
docs[:5]

[Document(metadata={'producer': 'PDF CoDe 2018 4.7111.7111 (c) 2002-2018 European Commission', 'creator': 'PDF CoDe 2018 4.7111.7111 (c) 2002-2018 European Commission', 'creationdate': '2021-04-22T15:27:19+02:00', 'moddate': '2021-04-22T15:27:19+02:00', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'source': 'The-AI-Act.pdf', 'total_pages': 108, 'page': 0, 'page_label': '1'}, page_content='EN   EN \n \n \n \nEUROPEAN \nCOMMISSION  \nBrussels, 21.4.2021  \nCOM(2021) 206 final \n2021/0106 (COD) \n \nProposal for a \nREGULATION OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL \nLAYING DOWN HARMONISED RULES ON ARTIFICIAL INTELLIGENCE \n(ARTIFICIAL INTELLIGENCE ACT) AND AMENDING CERTAIN UNION \nLEGISLATIVE ACTS \n{SEC(2021) 167 final} - {SWD(2021) 84 final} - {SWD(2021) 85 final}'),
 Document(metadata={'producer': 'PDF CoDe 2018 4.7111.7111 (c) 2002-2018 European Commission', 'creator': 'PDF CoDe 2018 4.7111.7111 (c) 2002-2018 European Commission', 'creationdate': '2021-04-22T15:27

In [ ]:
print(docs[0].page_content)                  # Text from first page
print(f"Metadata: {docs[0].metadata}")       # {'source': 'The-AI-Act.pdf', 'page': 0}

EN   EN 
 
 
 
EUROPEAN 
COMMISSION  
Brussels, 21.4.2021  
COM(2021) 206 final 
2021/0106 (COD) 
 
Proposal for a 
REGULATION OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL 
LAYING DOWN HARMONISED RULES ON ARTIFICIAL INTELLIGENCE 
(ARTIFICIAL INTELLIGENCE ACT) AND AMENDING CERTAIN UNION 
LEGISLATIVE ACTS 
{SEC(2021) 167 final} - {SWD(2021) 84 final} - {SWD(2021) 85 final}
Metadata: {'producer': 'PDF CoDe 2018 4.7111.7111 (c) 2002-2018 European Commission', 'creator': 'PDF CoDe 2018 4.7111.7111 (c) 2002-2018 European Commission', 'creationdate': '2021-04-22T15:27:19+02:00', 'moddate': '2021-04-22T15:27:19+02:00', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'source': 'The-AI-Act.pdf', 'total_pages': 108, 'page': 0, 'page_label': '1'}


In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # Target size: ~500 characters per chunk
    chunk_overlap=100    # Overlap: 100 chars between consecutive chunks
)

chunks = text_splitter.split_documents(docs)
print(len(chunks))

851


In [19]:
chunks[:5]

[Document(metadata={'producer': 'PDF CoDe 2018 4.7111.7111 (c) 2002-2018 European Commission', 'creator': 'PDF CoDe 2018 4.7111.7111 (c) 2002-2018 European Commission', 'creationdate': '2021-04-22T15:27:19+02:00', 'moddate': '2021-04-22T15:27:19+02:00', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'source': 'The-AI-Act.pdf', 'total_pages': 108, 'page': 0, 'page_label': '1'}, page_content='EN   EN \n \n \n \nEUROPEAN \nCOMMISSION  \nBrussels, 21.4.2021  \nCOM(2021) 206 final \n2021/0106 (COD) \n \nProposal for a \nREGULATION OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL \nLAYING DOWN HARMONISED RULES ON ARTIFICIAL INTELLIGENCE \n(ARTIFICIAL INTELLIGENCE ACT) AND AMENDING CERTAIN UNION \nLEGISLATIVE ACTS \n{SEC(2021) 167 final} - {SWD(2021) 84 final} - {SWD(2021) 85 final}'),
 Document(metadata={'producer': 'PDF CoDe 2018 4.7111.7111 (c) 2002-2018 European Commission', 'creator': 'PDF CoDe 2018 4.7111.7111 (c) 2002-2018 European Commission', 'creationdate': '2021-04-22T15:27

In [17]:
chunked_text = [chunk.page_content for chunk in chunks]
chunked_text[0]

'EN   EN \n \n \n \nEUROPEAN \nCOMMISSION  \nBrussels, 21.4.2021  \nCOM(2021) 206 final \n2021/0106 (COD) \n \nProposal for a \nREGULATION OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL \nLAYING DOWN HARMONISED RULES ON ARTIFICIAL INTELLIGENCE \n(ARTIFICIAL INTELLIGENCE ACT) AND AMENDING CERTAIN UNION \nLEGISLATIVE ACTS \n{SEC(2021) 167 final} - {SWD(2021) 84 final} - {SWD(2021) 85 final}'

### Embedding the Documents


In [16]:
from sentence_transformers import SentenceTransformer, util

sentences = ["I'm happy", "I'm full of happiness"]
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Compute embedding for both sentences
embedding_1 = model.encode(sentences[0], convert_to_tensor=True)
embedding_2 = model.encode(sentences[1], convert_to_tensor=True)

In [17]:
embedding_1.shape

torch.Size([384])

In [18]:
util.pytorch_cos_sim(embedding_1, embedding_2)

tensor([[0.8367]], device='cuda:0')

In [19]:
embedding_1 @ embedding_2

tensor(0.8367, device='cuda:0')

In [20]:
import torch

torch.dot(embedding_1, embedding_2)

tensor(0.8367, device='cuda:0')

In [23]:
chunk_embeddings = model.encode(chunked_text, convert_to_tensor=True)

In [24]:
chunk_embeddings.shape

torch.Size([854, 384])

## Retrieval

In [25]:
def search_documents(query, top_k=5):
    # Encode the query into a vector
    query_embedding = model.encode(query, convert_to_tensor=True)

    # Calculate cosine similarity between the query and all document chunks
    similarities = util.pytorch_cos_sim(query_embedding, chunk_embeddings)

    # Get the top k most similar chunks
    top_k_indices = similarities[0].topk(top_k).indices

    # Retrieve the corresponding document chunks
    results = [chunked_text[i] for i in top_k_indices]

    return results

In [ ]:
print(search_documents("What are prohibited ai practices?", top_k=2))

['TITLE  II \nPROHIBITED  ARTIFICIAL  INTELLIGENCE  PRACTICES  \nArticle 5  \n1. The following artificial intelligence practices shall be prohibited:  \n(a) the placing on the market, putting into service or use of an A I system that \ndeploys subliminal techniques beyond a person’s consciousness in order to \nmaterially distort a person’s behaviour in a manner that causes or is likely to \ncause that person or another person physical or psychological harm;',
 'low or minimal risk. The list of prohibited practices in Title II comprises all those AI systems \nwhose use is considered unacceptable as contravening Unio n values, for instance by violating \nfundamental rights. The prohibitions covers practices that have a significant potential to \nmanipulate persons  through subliminal techniques beyond their consciousness or exploit']

## Generation

In [27]:
from transformers import pipeline

from genaibook.core import get_device

device = get_device()
generator = pipeline(
    "text-generation", model="HuggingFaceTB/SmolLM-135M-Instruct", device=device
)

In [28]:
def generate_answer(query):
    # Retrieve relevant chunks
    context_chunks = search_documents(query, top_k=2)

    # Combine the chunks into a single context string
    context = "\n".join(context_chunks)

    # Generate a response using the context
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

    # Define the context to be passed to the model
    system_prompt = (
        "You are a friendly assistant that answers questions about the AI Act. "
        "If the user is not making a question, you can ask for clarification"
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    response = generator(messages, max_new_tokens=300)
    return response[0]["generated_text"][2]["content"]

In [29]:
answer = generate_answer("What are prohibited ai practices in the EU act?")
print(answer)

The EU Act prohibits the use of artificial intelligence practices that are harmful to individuals, such as:

* The placing on the market, putting into service or use of an A I system that is subliminal, that is, it is not intended to be used for any purpose other than to deceive or manipulate individuals.
* The use of A I systems that are designed to deceive or manipulate individuals, such as those used in advertising, marketing, or customer service.
* The use of A I systems that are designed to manipulate individuals, such as those used in surveillance or monitoring.

The EU Act prohibits the use of A I systems that are designed to deceive or manipulate individuals, such as those used in advertising, marketing, or customer service.

The EU Act prohibits the use of A I systems that are designed to deceive or manipulate individuals, such as those used in advertising, marketing, or customer service.

The EU Act prohibits the use of A I systems that are designed to deceive or manipulate i